# Microsoft Agent Framework — Azure OpenAI (Responses API)

For dis code sample, you go use **Microsoft Agent Framework (MAF)** make you create one simple agent supported by **Azure OpenAI** using **Responses API**.

> **Migration note:** Dis sample before e dey use Semantic Kernel with GitHub Models. E don migrate go Microsoft Agent Framework, and GitHub Models (wey dem don stop, e go retire July 2026) don change to Azure OpenAI, wey dey support Responses API. The `OpenAIChatClient` wey dey MAF dey target Azure OpenAI stable `/openai/v1/` endpoint and e dey use Responses API as default.

The main reason for dis sample na to show the steps wey dem go later use later for other code samples when dem dey implement different agentic patterns.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Import di Python Packages Wey You Need


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Di Define Tool

For inside Microsoft Agent Framework, **tool** na simple Python function wey get `@tool` decorator wey di agent fit call. Below na how we define tool wey go give random vacation place and no go dey repeat di one wey e give before.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Dewam di Agent

Hia, we dey dew di Agent wey get di name `TravelAgent`.

For dis example, we dey use very basic instructions. Feel free to change dis instructions make you see how di agent go behave different.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Run Agent

Now we fit run the agent. We create one `AgentSession` so the agent go remember the conversation through the different turns, then we send two `user_inputs`. The first one ask for trip; the second one talk say the user no like the suggestion and ask for another one — the agent dey use the session history plus the `get_random_destination` tool to respond.

You fit change these messages to see how the agent go react differently. The responses dey **streamed** token-by-token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
